# CS4207 Coding Challenge, Group 24: forecasting taxi demand in parallel

Fernando Furdui, Cormac Allen, James Buckley

**Problem.** Forecast how many NYC yellow-taxi pickups happen in each of the 263 taxi zones in each hour, so cabs can be
positioned before demand arrives.

**Data.** NYC TLC yellow taxi trip records, January to April 2026: `yellow_tripdata_2026-01.parquet` to
`yellow_tripdata_2026-04.parquet` (14.9 million trips), from https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page.
The notebook reads them from `/home/cs4207/data/` on Rockhill and from `data/` anywhere else (`config.py` picks).

**How to run.** `pip install -r requirements.txt`, then *Restart & Run All*. On Rockhill: `bash run_rockhill.sh`.
Every setting (paths, thresholds, features, model, worker counts, repeats, seeds) is in `config.py`; the code is in `src/`.

| Section | What | Rubric criterion |
|---|---|---|
| 0 | Setup: environment and hardware | 6 |
| 1 | Cleaning | 1 |
| 2 | Data parallelism | 3 |
| 3 | Features and time-based splits | 1, 4 |
| 4 | Gradient computation (task parallelism) | 4 |
| 5 | Hyperparameter search (task parallelism) | 4 |
| 6 | Speedup and Amdahl's law | 2 |
| 7 | Non-determinism and fixes | 5 |
| 8 | Final test results | 4 |
| 9 | Numbers for the poster and run time | 6, 7 |

## 0. Setup

In [ ]:
import config   # has to be imported first, it sets numpy to 1 thread per process
import os, platform, sys, time
import matplotlib, numpy as np, pandas as pd, pyarrow, threadpoolctl
from src import cleaning as C
from src.features import (FEATURES, build_features, month_rows, fold_boundaries, to_arrays,
                          fit_scaler, apply_scaler)
from src.model import (make_shards, train_sequential, train_parallel, shard_gradient, take_step, scores,
                       error_scores, make_grid, search_sequential, search_parallel, best_trial,
                       load_imbalance)
from src.benchmark import (speedup_table, plot_speedup, time_it, start_empty_pool, round_trip_seconds,
                           payload_mb)

start_time = time.perf_counter()

cores = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
print("Python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__,
      "| pyarrow", pyarrow.__version__,
      "| matplotlib", matplotlib.__version__, "| threadpoolctl", threadpoolctl.__version__)
print("Machine:", platform.node(), "|", platform.processor() or platform.machine(), "|", cores,
      "cores | GPU: none used (CPU only) | on Rockhill:", config.ON_ROCKHILL)
print("Workers:", config.WORKER_COUNTS, "| timed repeats:", config.N_REPEATS, "| seed:", config.SEED)
print("Data folder:", config.DATA_DIR)
print("Files:", [f.name for f in config.DATA_FILES])

## 1. Cleaning

### 1.1 Look at the footer first
The footer is at the end of every Parquet file and tells us the row groups and the min/max pickup
time in each one, without reading any actual rows.

Two things stand out:
- every month is split into 4 row groups, which is where our 16 shards come from (4 months x 4)
- the rows aren't sorted by time. Row group 2 covers the whole month and the first one starts on 31 Dec,
  so filtering by date can't skip anything. Instead we save reading time by only loading the 6 columns we
  actually use, out of 20.

In [ ]:
footer = pd.concat([C.footer_summary(f) for f in config.DATA_FILES], ignore_index=True)
footer.to_csv(config.RESULTS_DIR / "footer_row_groups.csv", index=False)
print(f"Reading {len(config.COLUMNS)} of 20 columns: {config.COLUMNS}")
footer

### 1.2 Cleaning rules
Checked in this order (`rule_masks()` in `src/cleaning.py`):

| Rule | Dropped if | Why |
|---|---|---|
| outside_month | pickup isn't in that file's month | spill-over into Dec/next month, would get counted twice |
| invalid_zone | zone 264 or 265 | "Unknown" and "Outside NYC" in the zone lookup, nothing to forecast |
| non_positive_fare | fare is 0 or less | refunds and voids, down to -$2,555. Not real new trips |
| bad_distance | 0 miles or over 100 | 0 means it never moved. NYC trips are about 35 mi max, so over 100 is a broken meter (one says 269,097 mi) |
| bad_duration | under 1 minute or over 3 hours | meter started by mistake, or left running. Same limits as the Week 10 example |

**Flex Fare trips** (payment_type 0) are missing passenger count and rate code, but they're real rides and
about a quarter of all trips, so we keep them. We don't use those two columns anyway.

**Hours with no pickups** count as 0, they're not missing. We fill in every zone x hour so the lag
features later on look at the real previous hour.

**Zones** are labels, not numbers, so the model never gets the zone ID as a number.

In [ ]:
shards = C.list_shards()
t = time.perf_counter()
demand, shard_stats = C.build_demand_sequential(shards)
serial_s = time.perf_counter() - t
print(f"{len(shards)} shards cleaned and counted one after another in {serial_s:.1f} s")

columns = ["rows_in", *C.RULES, "rows_out", "flex_fare_kept"]
cleaning = shard_stats.groupby("month")[columns].sum()
cleaning.loc["total"] = cleaning.sum()
cleaning["kept_%"] = (100 * cleaning["rows_out"] / cleaning["rows_in"]).round(2)
cleaning.to_csv(config.RESULTS_DIR / "cleaning_before_after.csv")
cleaning

In [ ]:
total = cleaning.loc["total"].drop("kept_%").astype(int)
print(f"Before: {total.rows_in:,} trips   After: {total.rows_out:,} trips "
      f"({100 * total.rows_out / total.rows_in:.2f}% kept)")
print(f"Flex Fare trips kept: {total.flex_fare_kept:,} ({100 * total.flex_fare_kept / total.rows_out:.1f}% of clean trips)")
print(f"Zone-hour table: {len(demand):,} rows = {demand.zone.nunique()} zones x {demand.hour.nunique():,} hours")
print(f"Zone-hours with 0 pickups: {100 * (demand.trips == 0).mean():.1f}%")
demand.head()

### 1.3 Duplicates and rate codes
Some trips show up twice with the same pickup/drop-off time, zones and distance. In January 35,174 of
the 35,713 are a ride plus a negative-fare copy that refunds it. The fare rule drops the refund and keeps
the ride, and the few left over fail the distance or duration rules, so nothing is double counted.

`RatecodeID` is a category and is blank for Flex Fare trips. We only need pickup time and zone, so we
never read it.

In [ ]:
dup_counts, dup_example = C.duplicate_check(config.DATA_FILES[0], config.MONTHS[0])
print(dup_counts)
dup_example[[C.PICKUP, "PULocationID", "DOLocationID", "trip_distance", "fare_amount", "payment_type"]]

### 1.4 Train/test split
Split by time, never randomly: Jan to Mar to build the model, April to test it once at the end.
No feature uses anything from later than the hour it's predicting.

## 2. Data parallelism

**Shards:** one per row group, so 4 files x 4 = 16.
- they come straight out of the original files with `read_row_group()`, no need to rewrite anything
- each shard knows its own month, so it can do the spill-over check by itself
- 16 shards and 3 workers: a free worker grabs the next shard (`chunksize=1`). That helps because the last
  row group in each file is smaller.

**Same function on every shard:** `clean_and_count_shard()` reads, cleans and counts.

**Putting it back together:** sort by shard id, then add up the counts. A zone-hour can show up in more than
one shard, so we add instead of overwriting. Counts are whole numbers, so the totals come out exact.

In [ ]:
print("Rows per shard:")
print(shard_stats.rows_in.describe()[["min", "mean", "max"]].round(0).to_string())

t = time.perf_counter()
demand_par, stats_par = C.build_demand_parallel(shards, workers=config.MAX_WORKERS)
parallel_s = time.perf_counter() - t

# parallel should give exactly the same table and the same cleaning numbers
assert demand_par.equals(demand) and stats_par.equals(shard_stats)
print(f"\nParallel ({config.MAX_WORKERS} workers): {parallel_s:.1f} s   serial: {serial_s:.1f} s "
      "(the serial run also had to read the files from disk the first time)")
print("Parallel result identical to serial: True")

Each worker only ever holds one row group (about 1M rows, 6 columns), never a full month, and sends back a
small table of counts. If a shard was ever too big for memory, `iter_batches()` could break it up further.

## Save the table for the next part

In [ ]:
demand.to_parquet(config.RESULTS_DIR / "demand_zone_hour.parquet", index=False)
print(f"Saved {len(demand):,} zone-hours to results/demand_zone_hour.parquet")

In [ ]:
# Section 3 starts from Part 1's saved table, exactly as it would on a fresh machine.
demand = pd.read_parquet(config.RESULTS_DIR / "demand_zone_hour.parquet")
print(f"Demand table: {len(demand):,} rows = {demand['zone'].nunique()} zones x {demand['hour'].nunique():,} hours, "
      f"{demand['trips'].sum():,} pickups")

## 3. Features

The target is `trips` (pickups in this zone in this hour). Each (zone, hour) row gets 13 features:

| Group | Features | Meaning |
|---|---|---|
| Lag | `lag_1, lag_2, lag_3, lag_24, lag_48, lag_168` | this zone's demand 1, 2, 3 hours ago, and at the same hour 1 day, 2 days, 1 week ago |
| Rolling | `roll_mean_3, roll_mean_24, roll_std_24` | mean demand over the previous 3 and 24 hours, and how much it varied over the previous 24 hours |
| Calendar | `hour_of_day, dayofweek, is_weekend, is_holiday` | known from the clock and a holiday list |

**Why there is no leakage:** a row predicts the hour it sits on, so that hour's own pickups must never be a
feature. Every lag is a `shift(k)` of the same zone's own series with `k >= 1`, and the rolling windows are
computed on a series that was shifted by 1 first, so they stop at the *previous* hour. Calendar features
come from the clock, not from the data. The first week of January has no 168-hour history, so those rows are dropped.
The check below proves it: changing the pickups inside one hour does not change that hour's features.

In [ ]:
df = build_features(demand)
print(f"Rows before features: {len(demand):,}   after: {len(df):,}   "
      f"(dropped {len(demand) - len(df):,} rows from the first week, which has no 168-hour history)")
print(df[["hour", "zone", "trips"] + FEATURES].head(3).T.to_string())

In [ ]:
# Leakage check: add 1000 pickups to every zone in one hour, rebuild, compare that hour's features.
hour = df["hour"].iloc[len(df) // 2]
changed = demand.copy()
changed.loc[changed["hour"] == hour, "trips"] += 1000
df_changed = build_features(changed)

before = df[df["hour"] == hour][FEATURES].reset_index(drop=True)
after = df_changed[df_changed["hour"] == hour][FEATURES].reset_index(drop=True)
print("Features of that hour unchanged when its own pickups change:", before.equals(after))
assert before.equals(after)

### Time-based splits and scaling

* **Development** = January to March, **test** = April. April is not touched until section 8.
* **Rolling validation folds** (from `config.py`): each fold trains on *everything before* its start date and is scored on the
  14 days after it. The folds start on Feb 1, Feb 15, Mar 1 and Mar 15, so the model is always asked to forecast
  the future from the past, never the other way round. A setting's score is its average over the four folds.
* **Scaling:** features are standardised (subtract mean, divide by standard deviation). The mean and standard
  deviation come from the **training rows only** (inside each fold, and from January to March for the final model),
  so later months never leak into training. Scaled features also keep gradient descent stable.

In [ ]:
dev = month_rows(df, config.DEV_MONTHS)
test = month_rows(df, config.TEST_MONTHS)
X_dev_raw, y_dev, zones_dev = to_arrays(dev)
X_test_raw, y_test, _ = to_arrays(test)

folds = fold_boundaries(dev["hour"].to_numpy(), config.VAL_FOLD_STARTS, config.VAL_FOLD_DAYS)
fold_table = pd.DataFrame({
    "fold start": config.VAL_FOLD_STARTS,
    "train rows": [t for t, _ in folds],
    "validation rows": [v - t for t, v in folds],
})
print(fold_table.to_string(index=False))
print(f"\nDevelopment rows (Jan-Mar): {len(y_dev):,}   Test rows (Apr): {len(y_test):,}")

mean, std = fit_scaler(X_dev_raw)
X_dev = apply_scaler(X_dev_raw, mean, std)      # used for the final model and the section 4 demo
X_test = apply_scaler(X_test_raw, mean, std)

## 4. Gradient computation (task parallelism)

**Linear regression in one paragraph.** The model predicts demand as `x . w + b`. Gradient descent starts at
`w = 0, b = 0`, measures how wrong the predictions are, and moves `w` and `b` a small step (the *learning rate*) in
the direction that reduces the error. It repeats this `GD_ITERATIONS` times. The "direction" is the **gradient**:
`sum((prediction - actual) * x)` over the rows. An L2 penalty gently pulls `w` toward 0.

**Why this parallelises.** The gradient is a *sum over rows*, and a sum can be split. We split the rows into
**6 zone shards** (`N_GRADIENT_SHARDS`): each shard holds *all the hours* for a group of zones. Every zone has the same number of
hours, so the shards are almost the same size (263 zones / 6 shards) and each worker gets about the same amount of work. Each iteration:

1. **Broadcast:** send the current `w` and `b` to every worker.
2. **Compute:** each worker sums `(prediction - actual) * x` over its own shard. Shards are independent: a worker reads only its own rows and the shared `w`, `b`.
3. **All-reduce:** add the shard sums together **in shard order** and divide by the total row count.
4. **Update:** take one step on the single shared model.

The shard sums add up to the same gradient as one pass over all rows, so the parallel model equals the sequential one
(up to tiny floating-point rounding, because the additions happen in a different order).
The shards are the same for 1, 2 or 3 workers, so every worker count gives *identical* weights.

Timings here are one run each. Person 3's section 6 repeats them for proper benchmarking.

In [ ]:
LR, L2, N_ITER = config.DEFAULT_LR, config.DEFAULT_L2, config.GD_ITERATIONS

t0 = time.perf_counter()
w_seq, b_seq = train_sequential(X_dev, y_dev, LR, L2, N_ITER)
seq_seconds = time.perf_counter() - t0

shards = make_shards(X_dev, y_dev, zones_dev)       # 6 zone shards
print("Shard sizes (rows):", [len(s_y) for _, s_y in shards])

rows = [{"workers": "sequential", "seconds": seq_seconds, "speedup": 1.0,
         "max |w - w_seq|": 0.0, "matches sequential": True}]
for n_workers in config.WORKER_COUNTS:
    t0 = time.perf_counter()
    w_par, b_par = train_parallel(shards, LR, L2, N_ITER, n_workers)
    seconds = time.perf_counter() - t0
    rows.append({"workers": n_workers, "seconds": seconds, "speedup": seq_seconds / seconds,
                 "max |w - w_seq|": float(np.abs(w_par - w_seq).max()),
                 "matches sequential": bool(np.allclose(w_par, w_seq) and np.isclose(b_par, b_seq))})

gradient_table = pd.DataFrame(rows)
print(gradient_table.to_string(index=False))
assert gradient_table["matches sequential"].all()

**Reading the table.** Every parallel model matches the sequential one (differences around 1e-15 are rounding).
Speedup is small or below 1: the whole development table is only about half a million rows, so one gradient pass is a
few milliseconds, while every iteration has to send `w`, `b` and the shard results between processes and wait for the slowest shard.
Those overheads (communication, synchronisation each iteration, process start-up) are what Person 3 analyses with Amdahl's law.

**Load imbalance here:** the shards are equal-sized, so each worker does about the same work. With 6 shards on 3 workers
each worker takes 2 shards per iteration; with 2 workers, 3 each; but a worker that is briefly slower still holds up
the others, because step 3 needs every shard before the next iteration can start.

## 5. Hyperparameter search (task parallelism)

A *hyperparameter* is a setting chosen before training (not learned from data). We try a 3 x 3 grid from `config.py`:
learning rate in {0.03, 0.1, 0.2} and L2 strength in {0, 0.001, 0.1}. That is **9 trials**.
(The learning rates stay below `2 / largest eigenvalue` of the scaled feature matrix, about 0.24, or gradient descent diverges.)

**Why the trials are independent:** each trial trains its own models from scratch on the same development rows and
reads only its own two settings. No trial needs another trial's result, so they can run on separate workers at the same
time, in any order.

**Why validation folds, not April:** the best setting is the one with the lowest **mean absolute error (MAE) averaged over the four
validation folds**, all inside January to March. If we chose on April, the final test score would be flattered by our own
choice. April is used once, at the end (section 8).

In [ ]:
search_data = {"X": X_dev_raw, "y": y_dev}     # raw features: each fold fits its own scaler on its own train rows

t0 = time.perf_counter()
seq_results = search_sequential(search_data, folds, N_ITER)
search_seq_seconds = time.perf_counter() - t0

search_rows = [{"workers": "sequential", "seconds": search_seq_seconds, "speedup": 1.0,
                "same 9 results": True}]
par_results = {}
for n_workers in config.WORKER_COUNTS:
    t0 = time.perf_counter()
    results = search_parallel(search_data, folds, N_ITER, n_workers)
    seconds = time.perf_counter() - t0
    par_results[n_workers] = results
    same = [(r["lr"], r["l2"], r["val_mae"]) for r in results] == \
           [(r["lr"], r["l2"], r["val_mae"]) for r in seq_results]
    search_rows.append({"workers": n_workers, "seconds": seconds,
                        "speedup": search_seq_seconds / seconds, "same 9 results": same})

search_table = pd.DataFrame(search_rows)
print(search_table.to_string(index=False))
assert search_table["same 9 results"].all()

In [ ]:
trial_table = pd.DataFrame(seq_results)[["lr", "l2", "val_mae", "seconds"]]
print(trial_table.to_string(index=False))

best = best_trial(seq_results)
print()
print(f"Best on validation folds: lr={best['lr']}, l2={best['l2']}, mean validation MAE={best['val_mae']:.3f} pickups")

imbalance = load_imbalance(par_results[config.MAX_WORKERS])
print(f"Trial times with {config.MAX_WORKERS} workers: slowest {imbalance['slowest']:.2f}s, "
      f"average {imbalance['mean']:.2f}s, slowest/average = {imbalance['ratio']:.2f}")

**Load imbalance between tasks.** If every trial took the same time, 9 trials on 3 workers would be 3 equal rounds.
`slowest/average` above shows how uneven they were. Every trial runs the same number of iterations on the same data, so the
times are close; with early stopping or different model sizes, some trials would finish early and leave workers idle
while one long trial finished. The split also matters: 9 trials on 3 workers is 3 full rounds, but on 2 workers it is 4.5 rounds,
so one worker idles during the last trial.

**Determinism:** each trial starts from `w = 0` with no random numbers and a fixed set of folds, and `pool.map` returns
results in grid order, so the table is identical every run. A tie would go to the earlier grid entry.

## 6. Speedup and Amdahl's law

We time the same three jobs one after another (serial) and in parallel:

| Job | Kind of parallelism | Serial version | Parallel version |
|---|---|---|---|
| Cleaning (section 2) | data parallel, 16 row-group shards | `build_demand_sequential` | `build_demand_parallel` |
| Gradient (section 4) | task parallel, 6 zone shards, 300 iterations | the same 6 shard sums, in a loop | `train_parallel` |
| Grid search (section 5) | task parallel, 9 independent trials | `search_sequential` | `search_parallel` |

**How each number is measured** (`speedup_table()` in `src/benchmark.py`):
- **Warm-up:** each version runs once without being timed. That first run pays one-off costs (reading the files from disk into
  memory, loading libraries), which would otherwise land on whichever version happened to go first.
- **Repeats:** then it runs `N_REPEATS` more times and we report the mean and standard deviation. One timing can be thrown off by
  anything else the machine is doing; the spread shows how much to trust the mean.
- **Same work, same answer:** every parallel result is checked to be *exactly* equal to the serial one, or the cell stops.
- **The baseline is the plain serial function, not a pool with one worker.** The gap between those two rows is the cost of
  starting a worker process and copying the data to it.

**Speedup** S(n) = serial time / time with n workers. **Efficiency** E(n) = S(n) / n, the share of each worker we actually used
(1.0 is perfect). **Amdahl's law** says S(n) = 1 / ((1 - p) + p / n), where p is the fraction of the work that can run in parallel.
Solving for p gives the `amdahl_p` column: **p = (1 - 1/S) / (1 - 1/n)**. The best possible speedup with any number of workers is 1 / (1 - p).
Amdahl's law assumes parallel work costs nothing extra, so when parallel is *slower* than serial (S < 1) p comes out negative: that
means the overheads cost more than the parallel part saves.

Every setting runs 4 times, so this section takes several minutes.

In [ ]:
# Inputs for sections 6 and 7, built from what sections 1-5 already made.
cleaning_shards = C.list_shards()                       # Part 1's 16 row-group shards
grad_shards = make_shards(X_dev, y_dev, zones_dev)      # Part 2's 6 zone shards
print(f"Workers {config.WORKER_COUNTS}, {config.N_REPEATS} timed runs per setting after 1 untimed warm-up")

### 6.1 Cleaning (data parallelism)

In [ ]:
cleaning_table = speedup_table(C.build_demand_sequential, C.build_demand_parallel, cleaning_shards)
cleaning_table.to_csv(config.RESULTS_DIR / "speedup_cleaning.csv", index=False)
cleaning_table.round(3)

### 6.2 Gradient computation (task parallelism)

The serial version here adds up the **same 6 shard sums in the same order** as `train_parallel`, just without worker processes.
Section 4's `train_sequential` sums all rows in one go instead, and because floating-point addition depends on order
((a + b) + c is not always a + (b + c) in the last digit), its weights differ from the parallel ones by about 1e-14. With the
same order the two are identical bit for bit, so serial and parallel really do the same work.

In [ ]:
def train_sharded_sequential(shards, lr, l2, n_iter):
    """Serial twin of train_parallel: same shards, same shard-order sum, no worker processes."""
    # shortcut: defined here until it is added to src/model.py next to train_parallel
    n_total = sum(len(y) for _, y in shards)
    w, b = np.zeros(shards[0][0].shape[1]), 0.0
    for _ in range(n_iter):
        gw, gb = np.zeros_like(w), 0.0
        for X, y in shards:
            part_gw, part_gb, _ = shard_gradient(w, b, X, y)
            gw, gb = gw + part_gw, gb + part_gb
        w, b = take_step(w, b, gw / n_total, gb / n_total, lr, l2)
    return w, b


gradient_table = speedup_table(lambda shards: train_sharded_sequential(shards, LR, L2, N_ITER),
                               lambda shards, workers: train_parallel(shards, LR, L2, N_ITER, workers),
                               grad_shards)
gradient_table.to_csv(config.RESULTS_DIR / "speedup_gradient.csv", index=False)
gradient_table.round(3)

### 6.3 Hyperparameter search (task parallelism)

Each trial records how long it took (`seconds`), which is different on every run by nature, so that one field is left out when
the serial and parallel results are compared. The learning rate, L2 strength and validation MAE of all 9 trials must match exactly.

In [ ]:
def without_timing(results):
    return [{k: v for k, v in r.items() if k != "seconds"} for r in results]


search_table = speedup_table(lambda data, folds: without_timing(search_sequential(data, folds, N_ITER)),
                             lambda data, folds, workers: without_timing(search_parallel(data, folds, N_ITER, workers)),
                             search_data, folds)
search_table.to_csv(config.RESULTS_DIR / "speedup_search.csv", index=False)
search_table.round(3)

### 6.4 All three jobs together

In [ ]:
speedup_tables = {"cleaning (data parallel)": cleaning_table,
                  "gradient (task parallel)": gradient_table,
                  "grid search (task parallel)": search_table}
summary = pd.concat({name: t.set_index(["version", "workers"])[["mean_s", "std_s", "speedup", "efficiency", "amdahl_p"]]
                     for name, t in speedup_tables.items()})
print(summary.round(3).to_string())
plot_speedup(speedup_tables)

### 6.5 Measuring the overheads

Amdahl's law only explains the part of the work that stays serial. The rest of the gap to ideal speedup is **overhead**: extra
work that only exists because we went parallel. We measure the three biggest ones directly:

- **Process start-up:** start a pool of empty workers and shut it down again, with no data and no work.
- **Communication per round trip:** on an already-running pool, send 6 empty tasks and wait for the answers. The parallel gradient
  does this once per iteration (300 times), to send `w` out and get the shard sums back.
- **Data transfer:** how many MB each worker receives when it starts (every worker gets a copy of all the shards or all the search data).

In [ ]:
overheads = []
for n in config.WORKER_COUNTS:
    _, startup_times = time_it(start_empty_pool, n)
    overheads.append({"workers": n, "empty pool start-up (s)": np.mean(startup_times),
                      "round trip (ms)": 1000 * round_trip_seconds(n)})
overheads = pd.DataFrame(overheads)
overheads.to_csv(config.RESULTS_DIR / "overheads.csv", index=False)

print(f"Data copied into each worker at start-up: gradient shards {payload_mb(grad_shards):.0f} MB, "
      f"search data {payload_mb((search_data, folds, N_ITER)):.0f} MB")
print(f"Gradient: {N_ITER} iterations x {len(grad_shards)} tasks = {N_ITER * len(grad_shards):,} tasks sent to workers")
overheads.round(3)

**Where the parallel gradient's time goes.** If parallel were free, the time would be the serial time divided by the
number of workers. The table adds the overheads we measured on top of that, and whatever is left over is the overhead we
did not measure separately: copying the shards into every worker, and each iteration waiting for the slowest shard. A negative leftover means the serial run was slower than the same work inside a worker: machine noise, not a real saving (see the check in 6.6).

In [ ]:
grad_serial_s = gradient_table.mean_s[0]
breakdown = pd.DataFrame({
    "workers": overheads["workers"],
    "measured parallel (s)": [gradient_table.query("version == 'parallel' and workers == @n").mean_s.item()
                              for n in overheads["workers"]],
    "real work / n (s)": grad_serial_s / overheads["workers"],
    "process start-up (s)": overheads["empty pool start-up (s)"],
    f"{N_ITER} round trips (s)": overheads["round trip (ms)"] * N_ITER / 1000,
})
breakdown["rest: copying data, waiting (s)"] = breakdown["measured parallel (s)"] - breakdown.iloc[:, 2:5].sum(axis=1)
breakdown.to_csv(config.RESULTS_DIR / "gradient_time_breakdown.csv", index=False)
breakdown.round(3)

### 6.6 Why each job speeds up (or doesn't)

**Gradient: little or no speedup.** One iteration of real work is only a few milliseconds. In the breakdown above only one part
gets smaller as workers are added: the real work, divided over the workers. Everything else is overhead that stays the same or
grows with more workers: starting the processes, copying about 60 MB of shards into *each* worker, and 300 round trips (every
iteration sends `w` out and waits for all 6 shard sums to come back, so it also waits for the slowest one). With this little work
per iteration the overheads eat most of the gain, so speedup stays near 1, or below 1 when they outweigh it. That is also why
`amdahl_p` comes out small or negative: what limits this job is overhead, not a serial part of the code. More work per iteration
(more rows, a bigger model) would help; more workers would not.

**Grid search: the best speedup.** Each trial is several seconds of independent work (4 folds x 300 iterations), and a worker
only talks to the main process twice per trial: once to receive its settings, once to return its score. Start-up and data copying
are paid once and then hidden behind a lot of real work, so speedup grows with workers. It should still fall short of ideal,
because of that start-up and copying and because of **load imbalance**: 9 trials split evenly over 3 workers (3 rounds), but over
2 workers they take 5 rounds, the last one with a worker idle.

**Cleaning: in between.** Each shard is a big piece of work (a row group of about a million trips) and sends back only a small
table of counts, so communication is cheap. What limits it is reading the files (all workers share the same disk), starting the
processes, and the merge at the end, which can only run once every shard is done: the serial part in Amdahl's law.

**Check before trusting a run.** Efficiency above 1 (or p above 1) means parallel beat the ideal, which is very unlikely for these
jobs. It means the serial run was slowed down by something else: on a laptop, Windows moving the process to a slower core, or
another program. The large `std_s` on those rows shows the same thing. The final numbers come from Rockhill, where a job gets its
own cores.

## 7. Non-determinism: same code, same data, different answer

A parallel program is **non-deterministic** when running it twice on the same data can give different results. Here we
show three real sources in our own pipeline. For each one: two or more runs that differ, the fix, a re-run showing the
output is now identical (checked with `assert`), and what the fix does *not* cover.

| Source | Where it shows up | Fix |
|---|---|---|
| 7.1 Completion order | cleaning: shards finish in a different order each run | sort results by shard id before merging |
| 7.2 Floating-point sum order | gradient: shard sums added in a different order | always add them in shard order |
| 7.3 Thread count | numpy's maths library splits sums across threads | pin it to 1 thread per process (`config.py`) |

This section takes under a minute: none of the demos are timed.

### 7.1 Completion order

`collect_in_completion_order()` (in `src/cleaning.py`) collects the shard results with `as_completed`: in whatever order the
workers *finish*, which depends on how busy each core is at that moment. We run it three times and look at the order.

In [ ]:
order_runs = [C.collect_in_completion_order(cleaning_shards, config.MAX_WORKERS) for _ in range(3)]
for i, run in enumerate(order_runs, 1):
    print(f"run {i} finished the shards in this order: {[shard_id for shard_id, _, _ in run]}")

# Before the fix: glue the partial count tables together in the order they arrived.
glued = [pd.concat([counts for _, counts, _ in run], ignore_index=True) for run in order_runs]
order_differed = not all(g.equals(glued[0]) for g in glued)
print("\nBefore the fix, the glued tables differ between runs:", order_differed)

# The fix: merge_partials() sorts by shard id first, then adds up the counts.
merged = [C.merge_partials(run)[0] for run in order_runs]
order_fixed = all(m.equals(merged[0]) for m in merged)
print("After the fix (sort by shard id, then merge), all runs identical:", order_fixed)
assert order_fixed

**Fix and its limits.** Each result carries its shard id, and `merge_partials()` sorts by it before combining, so the
order the workers finish in no longer matters. `build_demand_parallel()` gets the same effect from `pool.map`, which returns
results in the order the shards were sent. The limits: the ids must be fixed before the work starts (here: file, then row
group), so a different sharding (another file layout, or `iter_batches` with another batch size) still gives a different
order, just a repeatable one. And sorting means waiting for *every* shard before merging, so results cannot be used as
they stream in. If all three runs above happened to finish in the same order, run the cell again: the point is that the
order is not guaranteed.

### 7.2 Floating-point sum order

Computers store numbers with about 16 significant digits, so every addition rounds a little, and the rounding depends on
the order: `(a + b) + c` and `a + (b + c)` can differ in the last digit. The parallel gradient adds up 6 shard sums every
iteration. First: how many different answers do the same 6 numbers give, over every order they could arrive in?

In [ ]:
from itertools import permutations

w0, b0 = np.zeros(X_dev.shape[1]), 0.0
shard_sums = [shard_gradient(w0, b0, X, y)[0] for X, y in grad_shards]   # the 6 shard sums of one iteration


def add_in_order(order):
    total = np.zeros_like(shard_sums[0])
    for i in order:
        total = total + shard_sums[i]
    return total


all_orders = [add_in_order(order) for order in permutations(range(len(shard_sums)))]
print(f"The same {len(shard_sums)} shard sums added in all {len(all_orders)} possible orders give "
      f"{len({s.tobytes() for s in all_orders})} different answers")
print(f"Largest difference from shard order: {max(np.abs(s - all_orders[0]).max() for s in all_orders):.1e}, "
      f"on numbers of size up to {np.abs(all_orders[0]).max():.1e}")

One gradient is off only in the last digits, but training repeats this 300 times and every step starts from the last
one, so the differences carry through to the final model. Below, two trainings add the shard sums in **arrival order**
(a random order stands in for whichever worker finishes first, as with `as_completed`), then two add them in
**shard order**, which is what `train_parallel` does: `pool.map` returns the results in shard order.

In [ ]:
def train_arrival_order(shards, rng):
    """train_sharded_sequential, but each iteration adds the shard sums in a random order: whichever finished first."""
    n_total = sum(len(y) for _, y in shards)
    w, b = np.zeros(shards[0][0].shape[1]), 0.0
    for _ in range(N_ITER):
        sums = [shard_gradient(w, b, X, y) for X, y in shards]
        gw, gb = np.zeros_like(w), 0.0
        for i in rng.permutation(len(shards)):
            gw, gb = gw + sums[i][0], gb + sums[i][1]
        w, b = take_step(w, b, gw / n_total, gb / n_total, LR, L2)
    return w, b


# Unseeded on purpose: each run gets its own arrival orders, like real workers would.
(w_a, b_a), (w_b, b_b) = (train_arrival_order(grad_shards, np.random.default_rng()) for _ in range(2))
sum_order_differed = not (np.array_equal(w_a, w_b) and b_a == b_b)
print(f"Before the fix, two trainings in arrival order differ: {sum_order_differed} "
      f"(largest weight difference {np.abs(w_a - w_b).max():.1e})")

(w_c, b_c), (w_d, b_d) = (train_sharded_sequential(grad_shards, LR, L2, N_ITER) for _ in range(2))
sum_order_fixed = np.array_equal(w_c, w_d) and b_c == b_d
print("After the fix (always shard order), two trainings identical:", sum_order_fixed)
assert sum_order_fixed

**Fix and its limits.** Adding in a fixed order (shard 0, then 1, ..., then 5) makes every run identical, and
`train_parallel` gets it for free because `pool.map` keeps the shard order. It makes the answer *repeatable*, not *more
accurate*: shard order is no more correct than any other order. It also only holds while the shards stay the same. With a
different `N_GRADIENT_SHARDS` the partial sums change, and so do the last digits. That is why the number of shards is fixed
in `config.py` and does not change with the number of workers, so 1, 2 or 3 workers all give the same model. For the same
reason, section 4's `train_sequential`, which adds all rows in one go, differs from the parallel version by about 1e-14.

### 7.3 Thread count

numpy hands matrix maths to a library (OpenBLAS) that can split one big sum across several threads and then add their
partial results together: the same rounding problem as 7.2, inside one line of numpy. `config.py` pins it to 1 thread.
Here we undo the pin for a moment (`threadpool_limits`) and compute the same gradient with different numbers of threads.
Each thread count is run twice.

In [ ]:
import os
from threadpoolctl import threadpool_info, threadpool_limits

w_test = np.linspace(-1, 1, X_dev.shape[1])            # any fixed weights will do
cores = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
thread_rows = []
for n_threads in sorted({1, 2, 4, cores}):
    with threadpool_limits(n_threads):                 # undo config.py's pin, only inside this block
        first = X_dev.T @ (X_dev @ w_test - y_dev)
        second = X_dev.T @ (X_dev @ w_test - y_dev)
    if n_threads == 1:
        one_thread = first
    thread_rows.append({"threads": n_threads, "same answer twice": np.array_equal(first, second),
                        "same as 1 thread": np.array_equal(first, one_thread),
                        "largest difference from 1 thread": np.abs(first - one_thread).max()})
thread_table = pd.DataFrame(thread_rows)
print(thread_table.to_string(index=False))

threads_differed = not thread_table["same as 1 thread"].all()
print("\nBefore the fix, some thread counts give a different answer:", threads_differed)
pinned = [lib["num_threads"] for lib in threadpool_info()]
threads_fixed = all(n == 1 for n in pinned)
print("After the fix (config.py), threads used by numpy's maths library:", pinned)
assert threads_fixed

**Fix and its limits.** `config.py` sets `OMP_NUM_THREADS`, `OPENBLAS_NUM_THREADS` and the other thread settings to 1
*before* numpy loads (`src/__init__.py` imports it first, so this holds however the code is started), and every worker
process inherits the setting. One thread means one fixed order of additions, on any machine. The answer can depend on the
exact thread count, not just "one or many", which is why it is pinned to exactly 1 rather than to "however many cores
there are". The limits: it only works for libraries that read these settings. A different numpy or OpenBLAS **version** can
still change the last digits, which is why `requirements.txt` pins versions. And each process gives up multi-threaded maths,
which costs nothing here because the parallelism comes from worker processes (3 workers each running a full set of threads
would overload the machine anyway).

### 7.4 Randomness: none, by design

Unseeded random numbers are the most common source of non-determinism, but this pipeline uses none: the model starts from
`w = 0`, the grid search tries a fixed list of 9 settings, the folds are fixed dates and the shards are fixed zone groups. The
one random draw in this notebook is the deliberate "arrival order" demo in 7.2. If random search or random start weights were
added, each task would get its own seed from `config.SEED` and its task id, never from the worker or process id
(`np.random.default_rng([config.SEED, task_id])`), so the result would not depend on which worker ran which task.

### 7.5 Summary

In [ ]:
nondeterminism = pd.DataFrame([
    {"source": "completion order", "differed before the fix": order_differed,
     "fix": "sort by shard id before merging", "identical after the fix": order_fixed},
    {"source": "floating-point sum order", "differed before the fix": sum_order_differed,
     "fix": "always add shard sums in shard order", "identical after the fix": sum_order_fixed},
    {"source": "thread count", "differed before the fix": threads_differed,
     "fix": "pin numpy to 1 thread (config.py)", "identical after the fix": threads_fixed},
])
nondeterminism.to_csv(config.RESULTS_DIR / "nondeterminism.csv", index=False)
nondeterminism

## 8. Final test results (April)

The best settings from section 5 train the final model on all development months (Jan to Mar), using the parallel gradient from
section 4. It is then scored **once** on April, which was never used for any decision.
The metrics are in pickups per zone-hour: **MAE** (average absolute error), **RMSE** (punishes big misses more) and
**WAPE** (total absolute error divided by total demand, a percentage that is easy to read).

Two simple baselines show whether the model is worth having: "same hour yesterday" (`lag_24`) and "same hour last week" (`lag_168`),
plus "always predict the training average".

In [ ]:
shards = make_shards(X_dev, y_dev, zones_dev)
w_final, b_final = train_parallel(shards, best["lr"], best["l2"], N_ITER, config.MAX_WORKERS)

final = scores(w_final, b_final, X_test, y_test)
lag24 = error_scores(test["lag_24"].to_numpy(), y_test)
lag168 = error_scores(test["lag_168"].to_numpy(), y_test)
average = error_scores(np.full(len(y_test), y_dev.mean()), y_test)

final_table = pd.DataFrame([
    {"model": "linear regression (parallel gradient)", **final},
    {"model": "baseline: same hour last week (lag_168)", **lag168},
    {"model": "baseline: same hour yesterday (lag_24)", **lag24},
    {"model": "baseline: training average", **average},
])
print(final_table.to_string(index=False))

**Is 300 steps enough?** Linear regression has an exact answer (least squares). It is not part of the model; it is only a check
that gradient descent has finished learning. If our result is about as good as the exact one, there is nothing more to gain from more iterations.

In [ ]:
A = np.c_[X_dev, np.ones(len(y_dev))]
solution = np.linalg.lstsq(A, y_dev, rcond=None)[0]
exact = scores(solution[:-1], solution[-1], X_test, y_test)
print(f"Test MAE: gradient descent {final['mae']:.3f}   exact least squares {exact['mae']:.3f}")

In [ ]:
# Numbers for the poster, written to a file so every poster number traces to this notebook.
from pathlib import Path
config.RESULTS_DIR.mkdir(exist_ok=True)
lines = [
    f"person2 target = pickups per zone per hour",
    f"person2 months: development {config.DEV_MONTHS}, test {config.TEST_MONTHS}",
    f"person2 rows development / test = {len(y_dev)} / {len(y_test)}",
    f"person2 features = {len(FEATURES)}: {', '.join(FEATURES)}",
    f"person2 validation folds = {config.VAL_FOLD_STARTS}, {config.VAL_FOLD_DAYS} days each",
    f"person2 gradient shards = {len(shards)} zone shards, iterations = {N_ITER}",
    f"person2 gradient sequential seconds = {seq_seconds:.3f}",
] + [f"person2 gradient {r['workers']} workers: {r['seconds']:.3f}s, speedup {r['speedup']:.2f}"
     for r in rows[1:]] + [
    f"person2 search sequential seconds = {search_seq_seconds:.3f}",
] + [f"person2 search {r['workers']} workers: {r['seconds']:.3f}s, speedup {r['speedup']:.2f}"
     for r in search_rows[1:]] + [
    f"person2 best lr = {best['lr']}, l2 = {best['l2']}, mean validation MAE = {best['val_mae']:.3f}",
    f"person2 trial time slowest/average = {imbalance['ratio']:.2f}",
    f"person2 test MAE = {final['mae']:.3f}, RMSE = {final['rmse']:.3f}, WAPE = {final['wape']:.1%}",
    f"person2 baseline lag_168 test MAE = {lag168['mae']:.3f}, RMSE = {lag168['rmse']:.3f}, WAPE = {lag168['wape']:.1%}",
    f"person2 baseline lag_24 test MAE = {lag24['mae']:.3f}, RMSE = {lag24['rmse']:.3f}, WAPE = {lag24['wape']:.1%}",
    f"person2 baseline average test MAE = {average['mae']:.3f}, RMSE = {average['rmse']:.3f}, WAPE = {average['wape']:.1%}",
    f"person2 exact least squares test MAE = {exact['mae']:.3f}",
]
(config.RESULTS_DIR / "person2_numbers.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

## 9. Numbers for the poster and run time

Every number on the poster is written to `results/poster_numbers.txt` by this one run, so each one can be traced back
to an output above. Part 2's own list (`results/person2_numbers.txt`, written in section 8) is included.

In [ ]:
def p3_lines():
    out = []
    for job, table in speedup_tables.items():
        for r in table[table.version == "parallel"].itertuples():
            out.append(f"speedup {job}, {r.workers} workers: {r.mean_s:.2f}s, speedup {r.speedup:.2f}, "
                       f"efficiency {r.efficiency:.2f}, Amdahl p {r.amdahl_p:.2f}")
        out.append(f"speedup {job}, serial: {table.mean_s[0]:.2f}s (mean of {config.N_REPEATS} after a warm-up)")
    for r in overheads.itertuples():
        out.append(f"overhead {r.workers} workers: empty pool start-up {r[2]:.3f}s, round trip {r[3]:.2f} ms")
    out.append(f"overhead data copied into each worker: gradient {payload_mb(grad_shards):.0f} MB")
    for r in nondeterminism.itertuples():
        out.append(f"non-determinism {r.source}: differed before fix {r[2]}, fix '{r.fix}', identical after {r[4]}")
    return out


run_minutes = (time.perf_counter() - start_time) / 60
poster = ([f"run: {platform.node()}, {cores} cores, no GPU, on Rockhill {config.ON_ROCKHILL}, "
           f"Python {sys.version.split()[0]}, notebook run time {run_minutes:.1f} min",
           f"data: {', '.join(f.name for f in config.DATA_FILES)}",
           f"cleaning trips before / after = {total.rows_in:,} / {total.rows_out:,} "
           f"({100 * total.rows_out / total.rows_in:.2f}% kept)",
           f"cleaning zone-hour table = {len(demand):,} rows, {len(cleaning_shards)} shards"]
          + (config.RESULTS_DIR / "person2_numbers.txt").read_text().splitlines()
          + p3_lines())
(config.RESULTS_DIR / "poster_numbers.txt").write_text("\n".join(poster) + "\n")
print("\n".join(poster))
print(f"\nTotal run time: {run_minutes:.1f} min")